### 1.0 Preprocessing for modeling pipeline

In [4]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

In [5]:
train_df = pd.read_csv("../data/processed/ckd_train.csv")
X_train = train_df.drop(columns='class')
y_train = train_df['class'].map({"ckd": 1, "notckd": 0})

test_df = pd.read_csv("../data/processed/ckd_test.csv")
X_test = test_df.drop(columns='class')
y_test = test_df['class'].map({"ckd": 1, "notckd": 0})

Preprocessing:
- boolean columns:
    1. Impute using most frequent
    2. One-hot encode to binary
- continuous column:
    1. impute using median
    2. standard scaler

In [6]:
numeric_features = ['bgr']
binary_features = ['htn', 'dm', 'appet']


In [7]:
binary_imputer = SimpleImputer(strategy="most_frequent")
one_hot_encoder = OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)
binary_transformer = make_pipeline(
    binary_imputer,
    one_hot_encoder
)

In [8]:
num_imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
numeric_transformer = make_pipeline(
    num_imputer,
    scaler
)

In [ ]:
preprocessor = make_column_transformer(
    (binary_transformer, binary_features),
    (numeric_transformer, numeric_features)
)

### 2.0 Modeling

In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_predict, FixedThresholdClassifier
from sklearn.metrics import fbeta_score, make_scorer, classification_report, precision_recall_curve, confusion_matrix
import numpy as np

In [42]:
lr = LogisticRegression()
lr_pipe = make_pipeline(preprocessor, lr)

param_grid = {
    'logisticregression__C': np.logspace(-4, 4, 17)
}

grid_search = GridSearchCV(
    estimator=lr_pipe,
    param_grid=param_grid,
    n_jobs=-1, 
    cv=5,
    scoring=make_scorer(fbeta_score, beta=2),
    return_train_score=True)

grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.",{'logisticregression__C': array([1.0000...00000000e+04])}
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","make_scorer(f...dict', beta=2)"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.

In [25]:
import altair as alt

def plot_train_cv(param_vals, train_scores, val_scores, xlabel, ylabel="F2 Score"):
    df = pd.DataFrame({
        "param": list(param_vals) * 2,
        "score": list(train_scores) + list(val_scores),
        "split": ["train"] * len(param_vals) + ["valid"] * len(param_vals),
    })

    return alt.Chart(df).mark_line(point=True).encode(
        x=alt.X("param:Q", scale=alt.Scale(type="log"), title=xlabel),
        y=alt.Y("score:Q", scale=alt.Scale(zero=False), title=ylabel),
        color=alt.Color("split:N", title=None),
        tooltip=["param", "split", alt.Tooltip("score:Q", format=".3f")],
    )

In [27]:
cv_results = grid_search.cv_results_
plot_train_cv(
    cv_results['param_logisticregression__C'],
    cv_results['mean_train_score'],
    cv_results['mean_test_score'],
    xlabel='C'
)

alt.Chart(...)

In [36]:
lr_pipe.set_params(logisticregression__C=0.01)
lr_pipe.fit(X_train, y_train)
print(classification_report(y_train, lr_pipe.predict(X_train)))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       113
           1       0.62      0.99      0.77       187

    accuracy                           0.62       300
   macro avg       0.31      0.50      0.38       300
weighted avg       0.39      0.62      0.48       300



A C value of 0.01 heavily restricts the model to the point that it can't use any of the features. It then just uses the intercept which results in predicting the majority class every time.

To avoid this problem we will set the C value to 1 when tuning the decision threshold.

In [ ]:
## Determine threshold that achieves at least 95% recall
lr_pipe.set_params(logisticregression__C=1)

# out-of-fold probability of CKD for each training row
proba = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, proba)

# highest threshold that still achieves recall >= 0.95
target = 0.95
idx = np.where(recall[:-1] >= target)[0][-1]
threshold = thresholds[idx]
print(threshold, precision[idx], recall[idx])

0.2919693687396527 0.7574468085106383 0.9518716577540107


In [ ]:
## Precision-Recall Curve with identified threshold
pr_df = pd.DataFrame({
    "recall": recall[:-1],
    "precision": precision[:-1],
    "threshold": thresholds,
})

curve = alt.Chart(pr_df).mark_line().encode(
    x=alt.X("recall:Q", title="Recall", scale=alt.Scale(domain=[0, 1])),
    y=alt.Y("precision:Q", title="Precision", scale=alt.Scale(domain=[0, 1])),
    tooltip=[alt.Tooltip(c, format=".3f") for c in ["threshold", "recall", "precision"]],
)

chosen = alt.Chart(pd.DataFrame({
    "recall": [recall[idx]],
    "precision": [precision[idx]],
    "label": [f"threshold = {threshold:.2f}"],
})).mark_point(size=100, filled=True, color="red").encode(
    x="recall:Q", y="precision:Q"
)

label = chosen.mark_text(align="left", dx=8, dy=-8).encode(text="label:N")

baseline = alt.Chart(pd.DataFrame({"y": [y_train.mean()]})).mark_rule(
    strokeDash=[4, 4], color="gray"
).encode(y="y:Q")

(curve + baseline + chosen + label).properties(
    title="Precision-recall curve (out-of-fold, training data)",
    width=400, height=300,
)

alt.LayerChart(...)

In [44]:
final_model = FixedThresholdClassifier(lr_pipe, threshold=0.29)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.43      0.55        37
           1       0.73      0.92      0.82        63

    accuracy                           0.74       100
   macro avg       0.75      0.68      0.68       100
weighted avg       0.74      0.74      0.72       100



In [47]:
# Show F2 score and confusion matrix
print(fbeta_score(y_test, y_pred, beta=2))
print(confusion_matrix(y_test, y_pred))

0.8761329305135952
[[16 21]
 [ 5 58]]


The final model achieves recall of 0.92 on the held-out test set with an acceptable precision of 0.73.

The corresponding F2-score is 0.876.